# GSEA of L2 cell type DEGs against Hallmark Pathways

In this notebook, we'll perform enrichment tests for DEGs generated for each L2 cell type in comparisons of subjects against Hallmark Pathways, which are a high-level collection of gene sets.

These sets are available in the `msigdbr` package.

## Load packages

hise: The Human Immune System Explorer R SDK package  
purrr: Functional programming tools  
dplyr: Dataframe handling functions  
tibble: modern data.frame structures  
fgsea: Fast Gene Set Enrichment Analysis  

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(msigdbr)
quiet_library(purrr)
quiet_library(dplyr)
quiet_library(tibble)
quiet_library(fgsea)
quiet_library(furrr)
quiet_library(nanoparquet)

Warning message:
“package ‘msigdbr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘furrr’ was built under R version 4.4.3”
Warning message:
“package ‘nanoparquet’ was built under R version 4.4.3”


In [2]:
plan(multisession, workers = 12)
parallel_options <- furrr_options(seed = 3030)

In [3]:
if(!dir.exists("output")) {
    dir.create("output")
}

## Load Hallmark sets

In [4]:
hallmark <- msigdbr(species = "human", collection = "H")

In [5]:
hallmark_list <- split(hallmark, hallmark$gs_name)
hallmark_list <- map(hallmark_list, "gene_symbol")

In [6]:
length(hallmark_list)

[1] 50

In [7]:
hallmark_df <- data.frame(
    pathway = names(hallmark_list),
    n_pathway_genes = map_int(hallmark_list, length),
    pathway_genes = map_chr(hallmark_list, paste, collapse = ";")
)

## Load DEG files

We'll read in the .csv files for L2 DEGs previously generated

In [8]:
search_id <- "administrative_llama"

In [9]:
project_files <- listFilesInProjectStores(
    storesToList = list("rds"),
    toDF = TRUE
)

In [10]:
diff_project_files <- project_files %>%
  filter(grepl(search_id, name)) %>%
  filter(grepl("results.+parquet", name))

In [11]:
diff_file <- cacheFiles(list(diff_project_files$id))

[1] "downloading fileID 8ba84453-daab-4feb-9dd1-6a53dfb1e775"


In [12]:
diff <- read_parquet(diff_file)

In [13]:
head(diff)

,cell_type,fg,n_fg,bg,n_bg,gene,log2FoldChange,padj,direction,baseMean,lfcSE,stat,pvalue,fgMean,bgMean,allMean,cytokine,mode
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,ASDC,4-1BBL,10,PBS,11,CASC15,-2.7537635,0.02259652,HigherInPBS,38.8842873,0.5865781,-4.6946234,2.670983e-06,0.8964057,1.22667677,1.06940482,4-1BBL,Pseudobulk
2,ASDC,4-1BBL,10,PBS,11,ABCC4,-2.3647689,0.57220290,HigherInPBS,16.6009709,0.6195918,-3.8166560,1.352726e-04,0.7111491,0.96029548,0.84165434,4-1BBL,Pseudobulk
3,ASDC,4-1BBL,10,PBS,11,DPM1,-0.5118884,0.99987097,HigherInPBS,1.9101485,0.9591461,-0.5336918,5.935548e-01,0.2410387,0.25267853,0.24713577,4-1BBL,Pseudobulk
4,ASDC,4-1BBL,10,PBS,11,SCYL3,0.5923244,0.99987097,HigherIn4-1BBL,0.8851524,1.4299703,0.4142214,6.787119e-01,0.1672061,0.09243638,0.12804102,4-1BBL,Pseudobulk
5,ASDC,4-1BBL,10,PBS,11,C1orf112,1.2273376,0.99987097,HigherIn4-1BBL,0.5843965,1.4972382,0.8197343,4.123676e-01,0.1040167,0.06852690,0.08542681,4-1BBL,Pseudobulk
6,ASDC,4-1BBL,10,PBS,11,FGR,1.3052727,0.99987097,HigherIn4-1BBL,3.1635302,0.9731675,1.3412621,1.798354e-01,0.3488986,0.26337174,0.30409883,4-1BBL,Pseudobulk


In [14]:
cyto_diff <- split(diff, gsub(" ", "-", paste0(diff$fg, "_vs_",diff$bg)))

In [15]:
type_diff <- map(
    cyto_diff,
    function(diff) {
        split(diff, diff$cell_type)
    }
)
type_diff <- unlist(type_diff, recursive = FALSE)

### Prepare DEG lists

To rank genes, we'll convert nomP to -log10(nomP), and incorporate the direction of differential expression by multiplying by the direction of effect size (sign(logFC) if logFC is available, and sign(coef_D) if not).

We'll convert direction to a numeric value (1 or -1) to enable this calculation.

In [16]:
type_diff <- map(
    type_diff,
    function(deg) {
        deg %>%
          mutate(direction = ifelse(
              !is.na(log2FoldChange),
              sign(log2FoldChange),
              sign(stat)
          ))
})

We also need to avoid nomP values of 0. These will cause NA values due to log transformation. We'll convert these to `1e-300` so that they have a non-zero value.

In [17]:
type_diff <- map(
    type_diff,
    function(deg) {
        deg %>%
          mutate(pvalue = ifelse(
              pvalue == 0,
              1e-300, # if zero, change to 1e-300
              pvalue # otherwise, keep the value
          ))
})

In [18]:
deg_list <- map(
    type_diff,
    function(deg) {
        deg %>%
          mutate(rank_val = -log10(pvalue) * direction) %>%
          arrange(desc(rank_val))
    }
)

In [19]:
rank_list <- map(
    deg_list,
    function(deg) {
        v <- deg$rank_val
        names(v) <- deg$gene
        v
    }
)

## Run GSEA

In [20]:
fgsea_res <- future_map(
    rank_list,
    function(ranks) {
        suppressWarnings(
            fgsea(
                pathways = hallmark_list,
                stats    = ranks,
                minSize  = 10,
                maxSize  = 1000
            )
        )
    },
    .options = parallel_options
)

### Format results

In [21]:
deg_meta <- map(
    deg_list,
    function(deg) {
        list(
            fg = deg$fg[1],
            bg = deg$bg[1],
            cytokine = deg$cytokine[1],
            cell_type = deg$cell_type[1]
        )
    }
)

In [22]:
deg_meta[[1]]

$fg
[1] "4-1BBL"

$bg
[1] "PBS"

$cytokine
[1] "4-1BBL"

$cell_type
[1] "ASDC"

In [23]:
names(hallmark_df)

[1] "pathway"         "n_pathway_genes" "pathway_genes"

In [24]:
formatted_fgsea_res <- map2_dfr(
    fgsea_res,
    deg_meta,
    function(res, meta) {
        res %>%
          mutate(
              leadingEdge = map_chr(leadingEdge, paste, collapse = ";"),
              fg = meta$fg,
              bg = meta$bg,
              cytokine = meta$cytokine,
              cell_type = meta$cell_type
          ) %>%
          left_join(hallmark_df, by = "pathway") %>%
          rename(nomP = pval,
                 adjP = padj,
                 n_leadingEdge = size) %>%
          select(fg, bg, cytokine, cell_type,
                 pathway,
                 #pathway_label, 
                 NES, nomP, adjP, 
                 n_leadingEdge, n_pathway_genes,
                 leadingEdge, pathway_genes) %>%
          arrange(desc(NES))

    }
)

In [25]:
head(formatted_fgsea_res)

fg,bg,cytokine,cell_type,pathway,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>
4-1BBL,PBS,4-1BBL,ASDC,HALLMARK_COAGULATION,1.3607150,0.07522124,0.3071534,34,138,GNB2;CTSB;LTA4H;CTSO;WDR1;KLF7;ARF4;CTSH;LGMN;ADAM9;FURIN,A2M;ACOX2;ADAM9;ANG;ANXA1;APOA1;APOC1;APOC2;APOC3;ARF4;BMP1;C1QA;C1R;C1S;C2;C3;C8A;C8B;C8G;C9;CAPN2;CAPN5;CASP9;CD9;CFB;CFD;CFH;CFI;CLU;COMP;CPB2;CPN1;CPQ;CRIP2;CSRP1;CTSB;CTSE;CTSH;CTSK;CTSO;CTSV;DCT;DPP4;DUSP14;DUSP6;F10;F11;F12;F13B;F2;F2RL2;F3;F8;F9;FBN1;FGA;FGG;FN1;FURIN;FYN;GDA;GNB2;GNG12;GP1BA;GP9;GSN;HMGCS2;HNF4A;HPN;HRG;HTRA1;ISCU;ITGA2;ITGB3;ITIH1;KLF7;KLK8;KLKB1;LAMP2;LEFTY2;LGMN;LRP1;LTA4H;MAFF;MASP2;MBL2;MEP1A;MMP1;MMP10;MMP11;MMP14;MMP15;MMP2;MMP3;MMP7;MMP8;MMP9;MSRB2;MST1;OLR1;P2RY1;PDGFB;PECAM1;PEF1;PF4;PLAT;PLAU;PLEK;PLG;PREP;PROC;PROS1;PROZ;PRSS23;RABIF;RAC1;RAPGEF3;RGN;S100A1;S100A13;SERPINA1;SERPINB2;SERPINC1;SERPINE1;SERPING1;SH2B2;SIRT2;SPARC;TF;TFPI2;THBD;THBS1;TIMP1;TIMP3;TMPRSS6;USP11;VWF;WDR1
4-1BBL,PBS,4-1BBL,ASDC,HALLMARK_P53_PATHWAY,1.2361106,0.10123457,0.3306996,112,200,PPP1R15A;SP1;TSPYL2;ZMAT3;RAD9A;BAX;TPRKB;ERCC5;PLXNB2;TOB1;IFI30;STOM;BLCAP;CYFIP2;RCHY1;MDM2;DRAM1;CCNK;APP;APAF1;BTG2;RXRA;KLF4,ABAT;ABCC5;ABHD4;ACVR1B;ADA;AEN;AK1;ALOX15B;ANKRA2;APAF1;APP;ATF3;BAIAP2;BAK1;BAX;BLCAP;BMP2;BTG1;BTG2;CASP1;CCND2;CCND3;CCNG1;CCNK;CCP110;CD81;CD82;CDH13;CDK5R1;CDKN1A;CDKN2A;CDKN2AIP;CDKN2B;CEBPA;CGRRF1;CLCA2;COQ8A;CSRNP2;CTSD;CTSF;CYFIP2;DCXR;DDB2;DDIT3;DDIT4;DEF6;DGKA;DNTTIP2;DRAM1;EI24;ELP1;EPHA2;EPHX1;EPS8L2;ERCC5;F2R;FAM162A;FAS;FBXW7;FDXR;FGF13;FOS;FOXO3;FUCA1;GADD45A;GLS2;GM2A;GPX2;H1-2;H2AC25;H2AJ;HBEGF;HDAC3;HEXIM1;HINT1;HMOX1;HRAS;HSPA4L;IER3;IER5;IFI30;IL1A;INHBB;IP6K2;IRAG2;IRAK1;ISCU;ITGB4;JAG2;JUN;KIF13B;KLF4;KLK8;KRT17;LDHB;LIF;MAPKAPK3;MDM2;MKNK2;MXD1;MXD4;NDRG1;NHLH2;NINJ1;NOL8;NOTCH1;NUDT15;NUPR1;OSGIN1;PCNA;PDGFA;PERP;PHLDA3;PIDD1;PITPNC1;PLK2;PLK3;PLXNB2;PMM1;POLH;POM121;PPM1D;PPP1R15A;PRKAB1;PRMT2;PROCR;PTPN14;PTPRE;PVT1;RAB40C;RACK1;RAD51C;RAD9A;RALGDS;RAP2B;RB1;RCHY1;RETSAT;RGS16;RHBDF2;RNF19B;RPL18;RPL36;RPS12;RPS27L;RRAD;RRP8;RXRA;S100A10;S100A4;SAT1;SDC1;SEC61A1;SERPINB5;SERTAD3;SESN1;SFN;SLC19A2;SLC35D1;SLC3A2;SLC7A11;SOCS1;SP1;SPHK1;ST14;STEAP3;STOM;TAP1;TAX1BP3;TCHH;TCN2;TGFA;TGFB1;TM4SF1;TM7SF3;TNFSF9;TNNI1;TOB1;TP53;TP63;TPD52L1;TPRKB;TRAF4;TRAFD1;TRIAP1;TRIB3;TSC22D1;TSPYL2;TXNIP;UPP1;VAMP8;VDR;VWA5A;WRAP73;WWP1;XPC;ZBTB16;ZFP36L1;ZMAT3;ZNF365
4-1BBL,PBS,4-1BBL,ASDC,HALLMARK_CHOLESTEROL_HOMEOSTASIS,1.1428476,0.21226415,0.5200472,51,74,ACTG1;ANXA5;GUSB;ATF5;CHKA,ABCA2;ACAT2;ACSS2;ACTG1;ADH4;ALCAM;ALDOC;ANTXR2;ANXA13;ANXA5;ATF3;ATF5;ATXN2;AVPR1A;CBS;CD9;CHKA;CLU;CPEB2;CTNNB1;CXCL16;CYP51A1;DHCR7;EBP;ECH1;ERRFI1;ETHE1;FABP5;FADS2;FASN;FBXO6;FDFT1;FDPS;GLDC;GNAI1;GPX8;GSTM2;GUSB;HMGCR;HMGCS1;HSD17B7;IDI1;JAG1;LDLR;LGALS3;LGMN;LPL;LSS;MAL2;MVD;MVK;NFIL3;NIBAN1;NSDHL;PCYT2;PDK3;PLAUR;PLSCR1;PMVK;PNRC1;PPARG;S100A11;SC5D;SCD;SEMA3B;SQLE;SREBF2;STARD4;STX5;TM7SF2;TMEM97;TNFRSF12A;TP53INP1;TRIB3
4-1BBL,PBS,4-1BBL,ASDC,HALLMARK_TGF_BETA_SIGNALING,1.1378586,0.26107226,0.5561974,39,54,PPP1R15A;FNTA;SKI;PPP1CA;RHOA;ID2;FKBP1A;SMAD7;CDK9;SLC20A1;TGIF1;FURIN;CTNNB1,ACVR1;APC;ARID4B;BCAR3;BMP2;BMPR1A;BMPR2;CDH1;CDK9;CDKN1C;CTNNB1;ENG;FKBP1A;FNTA;FURIN;HDAC1;HIPK2;ID1;ID2;ID3;IFNGR2;JUNB;KLF10;LEFTY2;LTBP2;MAP3K7;NCOR2;NOG;PMEPA1;PPM1A;PPP1CA;PPP1R15A;RAB31;RHOA;SERPINE1;SKI;SKIL;SLC20A1;SMAD1;SMAD3;SMAD6;SMAD7;SMURF1;SMURF2;SPTBN1;TGFB1;TGFBR1;TGIF1;THBS1;TJP1;TRIM33;UBE2D3;WWTR1;XIAP
4-1BBL,PBS,4-1BBL,ASDC,HALLMARK_COMPLEMENT,1.0091648,0.42718447,0.6774194,107,200,GNB2;CD55;GNAI3;GMFB;ANXA5;CTSB;CBLB;STX4;LTA4H;CTSO;USP16;CASP4;GRB2;PIK3CG;GNG2;SH2B3;DOCK4;SIRT6;SRC;CTSH;LGMN;ADAM9;PIK3R5,ACTN2;ADAM9;ADRA2B;AKAP10;ANG;ANXA5;APOA4;APOBEC3F;APOBEC3G;APOC1;ATOX1;BRPF3;C1QA;C1QC;C1R;C1S;C2;C3;C4BPB;C9;CA2;CALM1;CALM3;CASP1;CASP10;CASP3;CASP4;CASP5;CASP7;CASP9;CBLB;CCL5;CD36;CD40LG;CD46;CD55;CD59;CDA;CDH13;CDK5R1;CEBPB;CFB;CFH;CLU;COL4A2;CP;CPM;CPQ;CR1;CR2;CSRP1;CTSB;CTSC;CTSD;CTS

In [26]:
unique(formatted_fgsea_res$cytokine)

[1] "4-1BBL"          "ADSF"            "APRIL"           "BAFF"           
 [5] "C3a"             "C5a"             "CD27L"           "CD30L"          
 [9] "CD40L"           "CT-1"            "Decorin"         "EGF"            
[13] "EPO"             "FGF-beta"        "FLT3L"           "FasL"           
[17] "G-CSF"           "GDNF"            "GITRL"           "GM-CSF"         
[21] "HGF"             "IFN-alpha1"      "IFN-beta"        "IFN-epsilon"    
[25] "IFN-gamma"       "IFN-lambda1"     "IFN-lambda2"     "IFN-lambda3"    
[29] "IFN-omega"       "IGF-1"           "IL-1-alpha"      "IL-1-beta"      
[33] "IL-10"           "IL-11"           "IL-12"           "IL-13"          
[37] "IL-15"           "IL-16"           "IL-17A"          "IL-17B"         
[41] "IL-17C"          "IL-17D"          "IL-17E"          "IL-17F"         
[45] "IL-18"           "IL-19"           "IL-1Ra"          "IL-20"          
[49] "IL-21"           "IL-22"           "IL-23"           "IL-24"          
[53] "IL-26"           "IL-27"           "IL-2"            "IL-31"          
[57] "IL-32-beta"      "IL-33"           "IL-34"           "IL-35"          
[61] "IL-36-alpha"     "IL-36Ra"         "IL-3"            "IL-4"           
[65] "IL-5"            "IL-6"            "IL-7"            "IL-8"           
[69] "IL-9"            "LIF"             "LIGHT"           "LT-alpha1-beta2"
[73] "LT-alpha2-beta1" "Leptin"          "M-CSF"           "Noggin"         
[77] "OSM"             "OX40L"           "PRL"             "PSPN"           
[81] "RANKL"           "SCF"             "TGF-beta1"       "TL1A"           
[85] "TNF-alpha"       "TPO"             "TRAIL"           "TSLP"           
[89] "TWEAK"           "VEGF"

In [27]:
unique(formatted_fgsea_res$cell_type)

[1] "ASDC"               "CD14 monocyte"      "CD16 monocyte"     
 [4] "CD56bright NK cell" "CD56dim NK cell"    "DN T cell"         
 [7] "Effector B cell"    "ILC"                "MAIT"              
[10] "Memory B cell"      "Memory CD4 T cell"  "Memory CD8 T cell" 
[13] "Naive B cell"       "Naive CD4 T cell"   "Naive CD8 T cell"  
[16] "Plasma cell"        "Progenitor cell"    "Treg"              
[19] "cDC1"               "cDC2"               "gdT"               
[22] "pDC"

In [28]:
formatted_fgsea_res %>%
  filter(
      fg == "IFN-alpha1",
      bg == "PBS",
      cytokine == "IFN-alpha1",
      cell_type == "CD14 monocyte"
  ) %>%
  head()

fg,bg,cytokine,cell_type,pathway,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>
IFN-alpha1,PBS,IFN-alpha1,CD14 monocyte,HALLMARK_INTERFERON_ALPHA_RESPONSE,2.391508,1.006909e-13,2.466926e-12,92,97,CCRL2;RIPK2;IRF2;PARP12;GBP2;TRIM26;DHX58;LPAR6;OGFR;TRIM21;TRAFD1;TMEM140;TRIM25;IFITM1;PARP9;IL15;IRF9;PLSCR1;CMTR1;RSAD2;ISG20;PARP14;TRIM5;OASL;OAS1;HERC6;UBA7;TDRD7;EPSTI1;SLC25A28;IFIH1;IFI44L;USP18;CXCL11;IFI27;SELL;RNF31;HELZ2;SAMD9;STAT2;BATF2;ADAR;TAP1;GMPR;LGALS3BP;IFIT2;IRF7;HLA-C;MX1;IFI44;CMPK2;MOV10;IFITM2;ISG15;CASP1;TRIM14;IFIT3;TENT5A;NUB1;CNP;SP110;SAMD9L;IFITM3;CD47;GBP4;WARS1;ELF1;EIF2AK2;IL4R;PSMB8;LAP3;LY6E;DDX60,ADAR;B2M;BATF2;BST2;C1S;CASP1;CASP8;CCRL2;CD47;CD74;CMPK2;CMTR1;CNP;CSF1;CXCL10;CXCL11;DDX60;DHX58;EIF2AK2;ELF1;EPSTI1;GBP2;GBP4;GMPR;HELZ2;HERC6;HLA-C;IFI27;IFI30;IFI35;IFI44;IFI44L;IFIH1;IFIT2;IFIT3;IFITM1;IFITM2;IFITM3;IL15;IL4R;IL7;IRF1;IRF2;IRF7;IRF9;ISG15;ISG20;LAMP3;LAP3;LGALS3BP;LPAR6;LY6E;MOV10;MVB12A;MX1;NCOA7;NMI;NUB1;OAS1;OASL;OGFR;PARP12;PARP14;PARP9;PLSCR1;PNPT1;PROCR;PSMA3;PSMB8;PSMB9;PSME1;PSME2;RIPK2;RNF31;RSAD2;RTP4;SAMD9;SAMD9L;SELL;SLC25A28;SP110;STAT2;TAP1;TDRD7;TENT5A;TMEM140;TRAFD1;TRIM14;TRIM21;TRIM25;TRIM26;TRIM5;TXNIP;UBA7;UBE2L6;USP18;WARS1
IFN-alpha1,PBS,IFN-alpha1,CD14 monocyte,HALLMARK_INTERFERON_GAMMA_RESPONSE,2.319538,3.003891e-15,1.471907e-13,174,200,CDKN1A;HLA-A;BPGM;TNFAIP3;IL10RA;RIPK2;TNFAIP2;CMKLR1;IRF5;FAS;IRF2;PARP12;PFKP;CASP3;TRIM26;DHX58;OGFR;TRIM21;CD274;TRAFD1;LCP2;CD38;JAK2;TRIM25;IRF8;PTPN1;LATS2;ARID5B;ZBP1;IFIT1;SOD2;MTHFD2;ST8SIA4;IL15;LYSMD2;IRF9;PLSCR1;ST3GAL5;CMTR1;RSAD2;PML;NLRC5;ISG20;PARP14;OASL;SPPL2A;GCH1;NFKB1;HERC6;TDRD7;EPSTI1;RIPK1;SLC25A28;IFIH1;IFI44L;USP18;IDO1;CXCL11;IFI27;IL15RA;RNF31;HELZ2;CFB;STAT2;BATF2;ZNFX1;ADAR;TAP1;ICAM1;SLAMF7;SOCS3;LGALS3BP;IFIT2;IRF7;MX1;IFI44;CMPK2;CSF2RB;STAT3;AUTS2;IFITM2;OAS3;PDE4B;ISG15;CASP1;TRIM14;IFIT3;OAS2;TNFSF10;CASP4;SERPING1;RNF213;SP110;STAT4;SAMD9L;IFITM3;EIF4E3;GBP4;RIGI;WARS1;MX2;EIF2AK2;TOR1B;IL4R;PSMB8;LAP3;BTG1;IRF4;LY6E;APOL6;DDX60;SECTM1;RBCK1,ADAR;APOL6;ARID5B;ARL4A;AUTS2;B2M;BANK1;BATF2;BPGM;BST2;BTG1;C1R;C1S;CASP1;CASP3;CASP4;CASP7;CASP8;CCL2;CCL5;CCL7;CD274;CD38;CD40;CD69;CD74;CD86;CDKN1A;CFB;CFH;CIITA;CMKLR1;CMPK2;CMTR1;CSF2RB;CXCL10;CXCL11;CXCL9;DDX60;DHX58;EIF2AK2;EIF4E3;EPSTI1;FAS;FCGR1A;FGL2;FPR1;GBP4;GBP6;GCH1;GPR18;GZMA;HELZ2;HERC6;HIF1A;HLA-A;HLA-B;HLA-DMA;HLA-DQA1;HLA-DRB1;HLA-G;ICAM1;IDO1;IFI27;IFI30;IFI35;IFI44;IFI44L;IFIH1;IFIT1;IFIT2;IFIT3;IFITM2;IFITM3;IFNAR2;IL10RA;IL15;IL15RA;IL18BP;IL2RB;IL4R;IL6;IL7;IRF1;IRF2;IRF4;IRF5;IRF7;IRF8;IRF9;ISG15;ISG20;ISOC1;ITGB7;JAK2;KLRK1;LAP3;LATS2;LCP2;LGALS3BP;LY6E;LYSMD2;MARCHF1;MT2A;MTHFD2;MVP;MX1;MX2;MYD88;NAMPT;NCOA3;NFKB1;NFKBIA;NLRC5;NMI;NOD1;NUP93;OAS2;OAS3;OASL;OGFR;P2RY14;PARP12;PARP14;PDE4B;PELI1;PFKP;PIM1;PLA2G4A;PLSCR1;PML;PNP;PNPT1;PSMA2;PSMA3;PSMB10;PSMB2;PSMB8;PSMB9;PSME1;PSME2;PTGS2;PTPN1;PTPN2;PTPN6;RAPGEF6;RBCK1;RIGI;RIPK1;RIPK2;RNF213;RNF31;RSAD2;RTP4;SAMD9L;SAMHD1;SECTM1;SELP;SERPING1;SLAMF7;SLC25A28;SOCS1;SOCS3;SOD2;SP110;SPPL2A;SRI;SSPN;ST3GAL5;ST8SIA4;STAT1;STAT2;STAT3;STAT4;TAP1;TAPBP;TDRD7;TMT1B;TNFAIP2;TNFAIP3;TNFAIP6;TNFSF10;TOR1B;TRAFD1;TRIM14;TRIM21;TRIM25;TRIM26;TXNIP;UBE2L6;UPP1;USP18;VAMP5;VAMP8;VCAM1;WARS1;XAF1;XCL1;ZBP1;ZNFX1
IFN-alpha1,PBS,IFN-alpha1,CD14 monocyte,HALLMARK_TNFA_SIGNALING_VIA_NFKB,2.199177,2.186351e-10,3.571040e-09,127,200,NFKBIE;TRAF1;CDKN1A;PPP1R15A;CCRL2;SNN;TNIP2;IER5;B4GALT5;MCL1;TNFAIP3;BIRC3;RNF19B;RIPK2;MAP2K3;NFKB2;TNFAIP2;CD83;TNFAIP8;CD80;ATF3;KLF6;GADD45B;SPSB1;KLF9;SERPINB8;TANK;DUSP5;ABCA1;PTGER4;RELA;DRAM1;EGR2;SOD2;CFLAR;PANX1;CCNL1;PFKFB3;NINJ1;SAT1;GCH1;PDLIM5;NFKB1;FUT4;TSC22D1;IFIH1;CXCL11;IL15RA;TAP1;ICAM1;FOSL2;SOCS3;IFIT2;MXD1;TGIF1;RELB;PDE4B,ABCA1;ACKR3;AREG;ATF3;ATP2B1;B4GALT1;B4GALT5;BCL2A1;BCL3;BCL6;BHLHE40;BIRC2;BIRC3;BMP2;BTG1;BTG2;BTG3;CCL2;CCL20;CCL4;CCL5;CCN1;CCND1;CCNL1;CCRL2;CD44;CD69;CD80;CD83;CDKN1A;CEBPB;CEBPD;CFLAR;CLCF1;CSF1;CSF2;CXCL1;CXCL10;CXCL11;CXC

In [29]:
formatted_fgsea_res %>%
  filter(
      fg == "IL-6",
      bg == "PBS",
      cytokine == "IL-6",
      cell_type == "Naive CD4 T cell"
  ) %>%
  head()

fg,bg,cytokine,cell_type,pathway,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>
IL-6,PBS,IL-6,Naive CD4 T cell,HALLMARK_MYC_TARGETS_V1,2.274480,9.656107e-09,4.538370e-07,111,200,LDHA;NPM1;TARDBP;RAN;PPIA;SRSF3;HNRNPU;SYNCRIP;RPLP0;PCBP1;YWHAQ;FAM120A;G3BP1;FBL;EIF3D;SMARCC1;NAP1L1;RACK1;TCP1;SRSF7;RPS3;HNRNPR;HNRNPC;EIF1AX;RPL18;EIF4G2;EIF2S1;PA2G4;PABPC4;HSPD1;SSBP1;EIF3B;PSMD14;EIF4A1;EPRS1;HSP90AB1;SRSF2;PGK1;EIF4E;RPS6;TOMM70;CLNS1A;EEF1B2;DHX15;DDX18;SERBP1;UBA2;NCBP1;SET;IARS1;HNRNPD;CNBP;ABCE1,ABCE1;ACP1;AIMP2;AP3S1;APEX1;BUB3;C1QBP;CAD;CANX;CBX3;CCNA2;CCT2;CCT3;CCT4;CCT5;CCT7;CDC20;CDC45;CDK2;CDK4;CLNS1A;CNBP;COPS5;COX5A;CSTF2;CTPS1;CUL1;CYC1;DDX18;DDX21;DEK;DHX15;DUT;EEF1B2;EIF1AX;EIF2S1;EIF2S2;EIF3B;EIF3D;EIF3J;EIF4A1;EIF4E;EIF4G2;EIF4H;EPRS1;ERH;ETF1;EXOSC7;FAM120A;FBL;G3BP1;GLO1;GNL3;GOT2;GSPT1;H2AZ1;HDAC2;HDDC2;HDGF;HNRNPA1;HNRNPA2B1;HNRNPA3;HNRNPC;HNRNPD;HNRNPR;HNRNPU;HPRT1;HSP90AB1;HSPD1;HSPE1;IARS1;IFRD1;ILF2;IMPDH2;KARS1;KPNA2;KPNB1;LDHA;LSM2;LSM7;MAD2L1;MCM2;MCM4;MCM5;MCM6;MCM7;MRPL23;MRPL9;MRPS18B;MYC;NAP1L1;NCBP1;NCBP2;NDUFAB1;NHP2;NME1;NOLC1;NOP16;NOP56;NPM1;ODC1;ORC2;PA2G4;PABPC1;PABPC4;PCBP1;PCNA;PGK1;PHB1;PHB2;POLD2;POLE3;PPIA;PPM1G;PRDX3;PRDX4;PRPF31;PRPS2;PSMA1;PSMA2;PSMA4;PSMA6;PSMA7;PSMB2;PSMB3;PSMC4;PSMC6;PSMD1;PSMD14;PSMD3;PSMD7;PSMD8;PTGES3;PWP1;RACK1;RAD23B;RAN;RANBP1;RFC4;RNPS1;RPL14;RPL18;RPL22;RPL34;RPL6;RPLP0;RPS10;RPS2;RPS3;RPS5;RPS6;RRM1;RRP9;RSL1D1;RUVBL2;SERBP1;SET;SF3A1;SF3B3;SLC25A3;SMARCC1;SNRPA;SNRPA1;SNRPB2;SNRPD1;SNRPD2;SNRPD3;SNRPG;SRM;SRPK1;SRSF1;SRSF2;SRSF3;SRSF7;SSB;SSBP1;STARD7;SYNCRIP;TARDBP;TCP1;TFDP1;TOMM70;TRA2B;TRIM28;TUFM;TXNL4A;TYMS;U2AF1;UBA2;UBE2E1;UBE2L3;USP1;VBP1;VDAC1;VDAC3;XPO1;XPOT;XRCC6;YWHAE;YWHAQ
IL-6,PBS,IL-6,Naive CD4 T cell,HALLMARK_E2F_TARGETS,1.970138,2.872545e-04,3.375240e-03,63,200,RBBP7;RAD21;TACC3;RAN;SYNCRIP;LBR;TP53;NAP1L1;PAICS;LMNB1;TFRC;MMS22L;EIF2S1;PA2G4;ILF3;MSH2;SRSF2;RAD50,AK2;ANP32E;ASF1A;ASF1B;ATAD2;AURKA;AURKB;BARD1;BIRC5;BRCA1;BRCA2;BRMS1L;BUB1B;CBX5;CCNB2;CCNE1;CCP110;CDC20;CDC25A;CDC25B;CDCA3;CDCA8;CDK1;CDK4;CDKN1A;CDKN1B;CDKN2A;CDKN2C;CDKN3;CENPE;CENPM;CHEK1;CHEK2;CIT;CKS1B;CKS2;CNOT9;CSE1L;CTCF;CTPS1;DCK;DCLRE1B;DCTPP1;DDX39A;DEK;DEPDC1;DIAPH3;DLGAP5;DNMT1;DONSON;DSCC1;DUT;E2F8;EED;EIF2S1;ESPL1;EXOSC8;EZH2;GINS1;GINS3;GINS4;GSPT1;H2AX;H2AZ1;HELLS;HMGA1;HMGB2;HMGB3;HMMR;HNRNPD;HUS1;ILF3;ING3;IPO7;JPT1;KIF18B;KIF22;KIF2C;KIF4A;KPNA2;LBR;LIG1;LMNB1;LUC7L3;LYAR;MAD2L1;MCM2;MCM3;MCM4;MCM5;MCM6;MCM7;MELK;MKI67;MLH1;MMS22L;MRE11;MSH2;MTHFD2;MXD3;MYBL2;MYC;NAA38;NAP1L1;NASP;NBN;NCAPD2;NME1;NOLC1;NOP56;NUDT21;NUP107;NUP153;NUP205;ORC2;ORC6;PA2G4;PAICS;PAN2;PCNA;PDS5B;PHF5A;PLK1;PLK4;PMS2;PNN;POLA2;POLD1;POLD2;POLD3;POLE;POLE4;POP7;PPM1D;PPP1R8;PRDX4;PRIM2;PRKDC;PRPS1;PSIP1;PSMC3IP;PTTG1;RACGAP1;RAD1;RAD21;RAD50;RAD51AP1;RAD51C;RAN;RANBP1;RBBP7;RFC1;RFC2;RFC3;RNASEH2A;RPA1;RPA2;RPA3;RRM2;SHMT1;SLBP;SMC1A;SMC3;SMC4;SMC6;SNRPB;SPAG5;SPC24;SPC25;SRSF1;SRSF2;SSRP1;STAG1;STMN1;SUV39H1;SYNCRIP;TACC3;TBRG4;TCF19;TFRC;TIMELESS;TIPIN;TK1;TMPO;TOP2A;TP53;TRA2B;TRIP13;TUBB;TUBG1;UBE2S;UBE2T;UBR7;UNG;USP1;WDR90;WEE1;XPO1;XRCC6;ZW10
IL-6,PBS,IL-6,Naive CD4 T cell,HALLMARK_G2M_CHECKPOINT,1.728761,2.317702e-03,2.178640e-02,78,200,SFPQ;AMD1;RAD21;TACC3;HNRNPU;SYNCRIP;LBR;G3BP1;NCL;SMARCC1;LMNB1;HIF1A;CUL3;ILF3;SRSF10;RBM14,ABL1;AMD1;ARID4A;ATF5;ATRX;AURKA;AURKB;BARD1;BCL3;BIRC5;BRCA2;BUB1;BUB3;CASP8AP2;CBX1;CCNA2;CCNB2;CCND1;CCNF;CCNT1;CDC20;CDC25A;CDC25B;CDC27;CDC45;CDC6;CDC7;CDK1;CDK4;CDKN1B;CDKN2C;CDKN3;CENPA;CENPE;CENPF;CHAF1A;CHEK1;CHMP1A;CKS1B;CKS2;CTCF;CUL1;CUL3;CUL4A;CUL5;DBF4;DDX39A;DKC1;DMD;DR1;DTYMK;E2F1;E2F2;E2F3;E2F4;EFNA5;EGF;ESPL1;EWSR1;EXO1;EZH2;FANCC;FBXO5;FOXN3;G3BP1;GINS2;GSPT1;H2AX;H2AZ1;H2AZ2;H2BC12;HIF1A;HIRA;HMGA1;HMGB3;HMGN2;HMMR;HNRNPD;HNRNPU;HOXC10;HSPA8;HUS1;ILF3;INCENP;JPT1;KATNA1;KIF11;KIF15;KIF20B;KIF22;KIF23;KIF2C;KIF4A;KIF5B;KMT5A;KNL1;KPNA2;KPNB1;LBR;LIG3;LMNB1;MAD2L1;MAP3K20;MAPK14;MARCKS;MCM2;MCM3;MCM5;MCM6;MEIS1;MEIS2;MKI67;MNAT1;

## Write output file

Write the metadata as a .csv for later use. We remove `row.names` and set `quote = FALSE` to simplify the outputs and increase compatibility with other tools.

In [30]:
gsea_out_file <- paste0("output/parse_10m_cytokine_AIFI_L2_DESeq2_hallmark_gsea_res_nomP-rank_", Sys.Date(), ".tsv")
write.table(
    formatted_fgsea_res,
    gsea_out_file,
    sep = "\t",
    row.names = FALSE,
    quote = FALSE
)

In [31]:
study_space_uuid <- "07e17d55-03ae-41ad-812c-a94ed2b98c76"
title <- paste("Parse 10M Cytokine AIFI_L2 Pseudobulk DESeq2 Hallmark GSEA", Sys.Date())

In [32]:
search_id <- ids::adjective_animal()
search_id

[1] "asphyxiated_africanjacana"

In [33]:
in_list <- as.list(diff_project_files$id)
in_list

[[1]]
[1] "8ba84453-daab-4feb-9dd1-6a53dfb1e775"

In [34]:
out_list <- list(gsea_out_file)

In [35]:
out_list

[[1]]
[1] "output/parse_10m_cytokine_AIFI_L2_DESeq2_hallmark_gsea_res_nomP-rank_2025-10-23.tsv"

In [36]:
uploadFiles(
    files = out_list,
    fileTypes = list("csv"),
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    destination = search_id,
    doPrompt = FALSE
)

checking if conda env can compile...

attempting to build conda environment...

Environment created successfully

packing conda environment...



[1] "/home/workspace/environment/rscrna"
[1] "Cannot determine the current notebook."
[1] "1) /home/workspace/parse-10m-pbmc-cytokines/aifi_labels/08a-R_L2_censored_reactome_gsea_nomP-rank.ipynb"
[1] "2) /home/workspace/parse-10m-pbmc-cytokines/aifi_labels/07b-R_L2_matched_hallmark_gsea_nomP-rank.ipynb"
[1] "3) /home/workspace/parse-10m-pbmc-cytokines/aifi_labels/07a-R_L2_matched_reactome_gsea_nomP-rank.ipynb"


Please select (1-3)  2


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "098740c8-c4f9-4537-b881-cb6489831571"

$ProcessId
[1] "b048f135-c4e6-43b1-905d-aadcb53b01b0"

$WorkflowId
[1] "687ae0f7-3d18-4a8a-baa1-da5e484a6ebd"

$FileIds
$FileIds[[1]]
[1] "55a288aa-fa9a-4afa-9fe2-7290297eb804"

In [37]:
sessionInfo()

R version 4.4.2 (2024-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/rscrna/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

Random number generation:
 RNG:     L'Ecuyer-CMRG 
 Normal:  Inversion 
 Sample:  Rejection 
 
locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] nanoparquet_0.4.2 furrr_0.3.1       future_1.34.0     fgsea_1.32.2     
[5] tibble_3.2.1      dplyr_1.1.4       purrr_1.1.0       msigdbr_25.1.1   
[9] hise_2.16.0      

loaded via a namespace (and not attached):
 [1] gene